# Model

- $t_i$: Survival time
- $d_i$: Event indicator (1 is death, 0 is censored)
- $x_i \in \mathbb{R}^p$, where $p=13,859$: Gene expression vector
- $s(i)$ site label

## Weibull AFT model

$$
\log(t_i) = \mu + \beta \cdot x_i + \varepsilon_i
$$

- $\mu$ is a global intercept (the baseline log survival time)
- $\boldsymbol{\beta} \in \mathbb{R}^p$ is a vector of gene coefficients
- $\varepsilon_i​$ is a residual error term

And
$$
\varepsilon_i \in \text{Gumbel}
$$


Equivalently
$$
t_i \sim \text{Weibull}(\alpha, \lambda_i)
$$

where $\alpha$ is the shape parameter and $\lambda_i = \exp(\mu + \beta\cdot x_i)$

## Handling censored data

If the patient died at time $t$, then that is the approximate probability density function of time of death.

If the patient is still alive by time $t$, then that is the survival function.

So
$$
\mathcal{L}_i = \begin{cases}
f(t_i) & d_i = 1 \text{ (died)} \\
S(t_i) & d_i = 0 \text{ (censored)}
\end{cases}
$$

where $f$ is the Weibull probability density function and $S$ is the survival function 

## Priors

The number of covariates (13,859) is much bigger than the number of data points (495). But most genes are irrelevant, only a few have an effect. So we use an **horseshoe prior**

For gene $j$, we have

$$
\beta_j \sim \mathcal{N}(0, \tau^2\lambda_j^2)
$$

$$
\lambda_j \sim \text{Half-Cauchy}(1)
$$

$$
\tau \sim \text{Half-Student-t}(3, 0, \sigma_\tau)
$$

Here
- $\lambda_j$ is the local shrinkage for gene $j$
- $\tau$ is the global shrinkage 

The $\sigma_\tau$ is calibrated using Piironen & Vehtari formula:
$$
\sigma_\tau = \frac{p_0}{p-p_0}\cdot \frac{\sigma}{\sqrt{n}}
$$

where
- $p_0$ is the prior guess for the number of relevant genes
- $p$ is the total number of genes
- $n$ is the number of patients
- $\sigma$ is the residual

### Model A

$$
\log (t_i) = \mu + \sum_{j=1}^p \beta_j x_{ij} + \varepsilon_i
$$

where $\beta_j$ follows the horshoe prior


In [1]:
import pymc as pm
import numpy as np
import pandas as pd
import pytensor.tensor as pt

def build_model_a(df: pd.DataFrame, gene_cols: list, p0: int = 30,
                  slab_scale: float = 2.0, slab_df: float = 4.0) -> pm.Model:
    """
    Model A — Naive Weibull AFT with REGULARIZED horseshoe prior on gene
    coefficients (Piironen & Vehtari 2017). No site adjustment. Handles
    right-censoring. The slab (c2) caps large coefficients so the linear
    predictor can't blow up the Weibull likelihood -> avoids divergences.
    """
    # survival models require t > 0; a time of 0 makes log(t) and the Weibull
    # density undefined -> NaN gradients -> 100% divergences.
    n_bad = int((df["OS_time"] <= 0).sum())
    if n_bad:
        print(f"dropping {n_bad} patient(s) with OS_time <= 0")
    df = df[df["OS_time"] > 0]

    X = df[gene_cols].values
    t = df["OS_time"].values
    d = df["OS"].values.astype(int)

    n, p = X.shape

    # split into observed deaths and censored
    obs_mask = d == 1
    cens_mask = d == 0

    X_obs  = X[obs_mask]
    X_cens = X[cens_mask]
    t_obs  = t[obs_mask]
    t_cens = t[cens_mask]

    # horseshoe global-scale calibration (Piironen & Vehtari)
    sigma_est = np.std(np.log1p(t))
    sigma_tau = (p0 / (p - p0)) * (sigma_est / np.sqrt(n))

    with pm.Model() as model:
        # --- priors ---
        mu    = pm.Normal("mu", mu=6, sigma=2)
        alpha = pm.HalfNormal("alpha", sigma=1)

        # regularized horseshoe (non-centered)
        tau = pm.HalfStudentT("tau", nu=3, sigma=sigma_tau)        # global shrinkage
        lam = pm.HalfCauchy("lam", beta=1, shape=p)                # local shrinkage
        c2  = pm.InverseGamma("c2", alpha=slab_df / 2,             # slab (caps big coefs)
                              beta=(slab_df / 2) * slab_scale**2)
        lam_tilde = pt.sqrt(c2 * lam**2 / (c2 + tau**2 * lam**2))
        z    = pm.Normal("z", mu=0, sigma=1, shape=p)
        beta = pm.Deterministic("beta", z * tau * lam_tilde)

        # --- linear predictors ---
        eta_obs  = mu + pm.math.dot(X_obs,  beta)
        eta_cens = mu + pm.math.dot(X_cens, beta)

        # --- likelihood for deaths (observed events) ---
        # Weibull AFT: scale lambda = exp(eta)
        y_obs = pm.Weibull(
            "y_obs",
            alpha=alpha,
            beta=pm.math.exp(eta_obs),
            observed=t_obs
        )

        # --- likelihood for censored (survival function) ---
        # log S(t) = -(t / lambda)^alpha = -exp(alpha * (log t - eta))   [stable form, t > 0]
        log_sf = -pm.math.exp(alpha * (np.log(t_cens) - eta_cens))
        cens_ll = pm.Potential("cens_ll", log_sf.sum())

    return model

In [2]:
import glob
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import gseapy as gp

df = pd.read_parquet("../data/processed/LUSC.parquet")

# --- MSigDB Hallmark panel -> Ensembl -> intersect with matrix ---
hm = gp.get_library(name="MSigDB_Hallmark_2020", organism="Human")   # 50 sets
symbols = {g for genes in hm.values() for g in genes}

# symbol -> versioned ENSG from any STAR annotation TSV
tsv = glob.glob("../data/raw/LUSC/*/*.rna_seq.augmented_star_gene_counts.tsv")[0]
ref = pd.read_csv(tsv, sep="\t", skiprows=1)
ref = ref[ref["gene_id"].str.startswith("ENSG")]
sym2ens = dict(zip(ref["gene_name"], ref["gene_id"]))
hm_ens = {sym2ens[s] for s in symbols if s in sym2ens}

gene_cols = [c for c in df.columns if c.startswith("ENSG") and c in hm_ens]
print(f"hallmark genes used: {len(gene_cols)}  (of {len(symbols)} symbols)")
print(f"Patients: {len(df)}  Deaths: {int(df['OS'].sum())}  Censored: {int((df['OS']==0).sum())}")

model = build_model_a(df, gene_cols, p0=10)

with model:
    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.95,
        nuts_sampler="nutpie",          # Rust NUTS: faster, handles its own parallelism
        return_inferencedata=True,
    )

print(az.summary(trace, var_names=["mu", "alpha", "tau"], round_to=3))
print(f"Divergences: {int(trace.sample_stats.diverging.sum())}")

hallmark genes used: 3867  (of 4383 symbols)
Patients: 495  Deaths: 212  Censored: 283
dropping 1 patient(s) with OS_time <= 0


NUTS[nutpie]: [mu, alpha, tau, lam, c2, z]
INFO:pymc.sampling.mcmc:NUTS[nutpie]: [mu, alpha, tau, lam, c2, z]


Output()

        mean     sd  eti89_lb  eti89_ub  ess_bulk  ess_tail  r_hat  mcse_mean  \
mu     7.791  0.085     7.661     7.935  6041.554  3302.035  1.002      0.001   
alpha  0.889  0.050     0.813     0.971  7055.209  2907.922  1.000      0.001   
tau    0.000  0.000     0.000     0.000  1042.408  1436.132  1.002      0.000   

       mcse_sd  
mu       0.001  
alpha    0.000  
tau      0.000  
Divergences: 26


### Model B - with site intercepts

$$
\log (t_i) = \mu + \sum_{j=1}^p \beta_j x_{ij} + \mu_{s(i)} + \varepsilon_i
$$

$$
\mu_s \sim \mathcal{N}(0, \sigma^2_\text{site}), \text s = 1\cdots S
$$

$$
\sigma_\text{site} \sim \text{Half-Normal}(1)
$$